# 08 — Full Evaluation Pipeline

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Purpose:** Compare MARS against 8 baselines, run ablation study,
compute statistical significance (paired t-test + Wilcoxon, 5 seeds).

**Tables produced:**  
- Table 1: Overall comparison (MARS vs 8 baselines × 8 metrics)  
- Table 2: Ablation study (5 component removals)

In [ ]:
import sys, os
sys.path.insert(0, "..")
os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from scipy import stats as sp_stats
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    ndcg_score, average_precision_score,
)
import torch
import torch.nn as nn

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 12, "axes.titlesize": 13,
    "axes.labelsize": 12, "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
SEEDS = [42, 123, 456, 789, 1024]
NUM_TAGS = 293

import logging
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s")

print("Libraries loaded.")

## 1. Load Data & Prepare Splits

In [ ]:
from data.loader import EdNetLoader
from data.preprocessor import EdNetPreprocessor

loader = EdNetLoader(data_dir="data/raw")
interactions = loader.load_interactions(sample_users=1000)
questions = loader.questions
lectures = loader.lectures

preprocessor = EdNetPreprocessor(output_dir="data/processed", splits_dir="data/splits")
df_clean = preprocessor.clean(interactions)
df_feat = preprocessor.engineer_features(df_clean)
splits = preprocessor.chronological_split(df_feat)

train_df = splits["train"]
val_df = splits["val"]
test_df = splits["test"]

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"Users in test: {test_df['user_id'].nunique()}")

## 2. Evaluation Helpers

In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline 5: DKT (Deep Knowledge Tracing — standard LSTM)
# ═══════════════════════════════════════════════════════════
class DKTModel(nn.Module):
    """Standard DKT: LSTM on (tag_id, correct) → P(correct|next tag)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        """tag_ids: (B,T), correct: (B,T,1) → (B, n_tags)."""
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        out, _ = self.lstm(x)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_dkt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                # Label: failure vector for next interaction
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = DKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_dkt(eval_pairs, dkt_model, seq_len=50):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
        corr = ctx["correct"].astype(float).values
        # Pad/truncate
        if len(tags) >= seq_len:
            tags = tags[-seq_len:]
            corr = corr[-seq_len:]
        else:
            tags = np.pad(tags, (seq_len - len(tags), 0))
            corr = np.pad(corr, (seq_len - len(corr), 0))

        with torch.no_grad():
            t_t = torch.tensor(tags, dtype=torch.long).unsqueeze(0)
            t_c = torch.tensor(corr, dtype=torch.float).unsqueeze(0).unsqueeze(-1)
            scores = dkt_model(t_t, t_c).squeeze(0).numpy()
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 6: SAKT (Self-Attentive Knowledge Tracing)
# ═══════════════════════════════════════════════════════════
class SAKTModel(nn.Module):
    """Simplified SAKT: self-attention on (tag_emb + correct), projected to d_model."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, d_model=64, n_heads=4, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.input_proj = nn.Linear(emb_dim + 1, d_model)
        self.pos_emb = nn.Embedding(64, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=hidden, dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        x = self.input_proj(x)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        pos = pos.clamp(0, 63)
        x = x + self.pos_emb(pos)
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, mask=mask)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_sakt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = SAKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_sakt(eval_pairs, sakt_model, seq_len=50):
    return baseline_dkt(eval_pairs, sakt_model, seq_len)  # same inference


print("Baselines 5-6 (DKT, SAKT) defined.")

## 3. Baseline Implementations

In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline 1: Random
# ═══════════════════════════════════════════════════════════
def baseline_random(eval_pairs, seed=42):
    rng = np.random.RandomState(seed)
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        s = rng.random(NUM_TAGS).astype(np.float32)
        ranked = np.argsort(s)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(s)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 2: Popular
# ═══════════════════════════════════════════════════════════
def baseline_popular(eval_pairs, train_df):
    # Count tag failure frequency in training set
    tag_fail_count = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        if not row["correct"]:
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    tag_fail_count[t] += 1
    # Normalise to [0, 1]
    tag_scores = tag_fail_count / (tag_fail_count.max() + 1e-10)
    ranked = np.argsort(tag_scores)[::-1].tolist()

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        preds.append(ranked)
        scores_list.append(tag_scores.astype(np.float32))
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 3: CF-only (ALS/SVD)
# ═══════════════════════════════════════════════════════════
def baseline_cf_only(eval_pairs, train_df, seed=42):
    from sklearn.decomposition import TruncatedSVD
    import scipy.sparse as sp

    # Build user x tag failure-rate matrix from train
    records = []
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                records.append((str(row["user_id"]), t, 1.0 - float(row["correct"])))

    rec_df = pd.DataFrame(records, columns=["user_id", "tag_id", "fail_rate"])
    agg = rec_df.groupby(["user_id", "tag_id"])["fail_rate"].mean().reset_index()

    users = sorted(agg["user_id"].unique())
    user_map = {u: i for i, u in enumerate(users)}

    rows_idx = [user_map[u] for u in agg["user_id"]]
    cols_idx = agg["tag_id"].values.astype(int)
    vals = agg["fail_rate"].values
    mat = sp.csr_matrix((vals, (rows_idx, cols_idx)), shape=(len(users), NUM_TAGS))

    n_comp = min(32, min(mat.shape) - 1)
    svd = TruncatedSVD(n_components=max(n_comp, 2), random_state=seed)
    user_factors = svd.fit_transform(mat)
    item_factors = svd.components_.T

    preds, scores_list, gt_list, gta_list = [], [], [], []
    rng = np.random.RandomState(seed)
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        if uid in user_map:
            scores = item_factors @ user_factors[user_map[uid]]
        else:
            scores = rng.random(NUM_TAGS).astype(np.float32) * 0.1
        scores = scores.astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 4: Content-only (SBERT similarity)
# ═══════════════════════════════════════════════════════════
def baseline_content_only(eval_pairs, train_df, questions_df, seed=42):
    """Score tags by avg difficulty in user's weak parts."""
    # Per-tag difficulty from training
    tag_difficulty = np.full(NUM_TAGS, 0.5)
    tag_counts = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                tag_difficulty[t] += (1.0 - float(row["correct"]))
                tag_counts[t] += 1
    mask = tag_counts > 0
    tag_difficulty[mask] /= tag_counts[mask]

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # User's weak tags from context
        user_acc = defaultdict(lambda: {"c": 0, "t": 0})
        for _, row in ctx.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    user_acc[t]["t"] += 1
                    if row["correct"]:
                        user_acc[t]["c"] += 1

        scores = np.copy(tag_difficulty)
        for t, s in user_acc.items():
            if s["t"] > 0:
                user_fail_rate = 1.0 - s["c"] / s["t"]
                scores[t] = 0.5 * scores[t] + 0.5 * user_fail_rate
        scores = scores.astype(np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Baselines 1-4 defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline 5: DKT (Deep Knowledge Tracing — standard LSTM)
# ═══════════════════════════════════════════════════════════
class DKTModel(nn.Module):
    """Standard DKT: LSTM on (tag_id, correct) → P(correct|next tag)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim + 1, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, n_tags)

    def forward(self, tag_ids, correct):
        """tag_ids: (B,T), correct: (B,T,1) → (B, n_tags)."""
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        out, _ = self.lstm(x)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_dkt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                # Label: failure vector for next interaction
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = DKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_dkt(eval_pairs, dkt_model, seq_len=50):
    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        tags = ctx["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
        corr = ctx["correct"].astype(float).values
        # Pad/truncate
        if len(tags) >= seq_len:
            tags = tags[-seq_len:]
            corr = corr[-seq_len:]
        else:
            tags = np.pad(tags, (seq_len - len(tags), 0))
            corr = np.pad(corr, (seq_len - len(corr), 0))

        with torch.no_grad():
            t_t = torch.tensor(tags, dtype=torch.long).unsqueeze(0)
            t_c = torch.tensor(corr, dtype=torch.float).unsqueeze(0).unsqueeze(-1)
            scores = dkt_model(t_t, t_c).squeeze(0).numpy()
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 6: SAKT (Self-Attentive Knowledge Tracing)
# ═══════════════════════════════════════════════════════════
class SAKTModel(nn.Module):
    """Simplified SAKT: self-attention on (tag_emb + correct)."""
    def __init__(self, n_tags=NUM_TAGS, emb_dim=32, n_heads=4, hidden=64):
        super().__init__()
        self.tag_emb = nn.Embedding(n_tags, emb_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(64, emb_dim + 1)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim + 1, nhead=n_heads,
            dim_feedforward=hidden, dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(emb_dim + 1, n_tags)

    def forward(self, tag_ids, correct):
        emb = self.tag_emb(tag_ids.clamp(0, NUM_TAGS - 1))
        x = torch.cat([emb, correct], dim=-1)
        B, T, D = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        pos = pos.clamp(0, 63)
        x = x + self.pos_emb(pos)
        # Causal mask
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        out = self.transformer(x, mask=mask)
        logits = self.fc(out[:, -1, :])
        return torch.sigmoid(logits)


def train_sakt(train_df, val_df, seed=42, epochs=15, seq_len=50):
    torch.manual_seed(seed)
    np.random.seed(seed)

    def build_seqs(df, sl=seq_len):
        seqs_t, seqs_c, labels = [], [], []
        for _, grp in df.groupby("user_id"):
            grp = grp.sort_values("timestamp")
            tags_arr = grp["tags"].apply(lambda x: parse_tags(x)[0] if parse_tags(x) else 0).values
            corr_arr = grp["correct"].astype(float).values
            for i in range(len(grp) - sl - 1):
                seqs_t.append(tags_arr[i:i+sl])
                seqs_c.append(corr_arr[i:i+sl])
                lbl = np.zeros(NUM_TAGS, dtype=np.float32)
                next_tags = parse_tags(grp.iloc[i+sl].get("tags", []))
                if not corr_arr[i+sl]:
                    for t in next_tags:
                        if 0 <= t < NUM_TAGS:
                            lbl[t] = 1.0
                labels.append(lbl)
        if not seqs_t:
            return None, None, None
        return (
            torch.tensor(np.array(seqs_t), dtype=torch.long),
            torch.tensor(np.array(seqs_c), dtype=torch.float).unsqueeze(-1),
            torch.tensor(np.array(labels), dtype=torch.float),
        )

    t_tags, t_corr, t_labels = build_seqs(train_df)
    if t_tags is None:
        return None

    model = SAKTModel()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCELoss()

    ds = torch.utils.data.TensorDataset(t_tags, t_corr, t_labels)
    dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True)

    model.train()
    for ep in range(epochs):
        for bt, bc, bl in dl:
            opt.zero_grad()
            pred = model(bt, bc)
            loss = criterion(pred, bl)
            loss.backward()
            opt.step()

    model.eval()
    return model


def baseline_sakt(eval_pairs, sakt_model, seq_len=50):
    return baseline_dkt(eval_pairs, sakt_model, seq_len)  # same inference


print("Baselines 5-6 (DKT, SAKT) defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Baseline 7: Binary-conf (MARS with binary correct/incorrect)
# Uses our LSTM but with conf_class always 0 (no 6-class info)
# ═══════════════════════════════════════════════════════════
def baseline_binary_conf(eval_pairs, train_df, seed=42):
    from agents.prediction_agent import PredictionAgent
    agent = PredictionAgent()
    # Train with zeroed confidence classes
    df = train_df.copy()
    if "confidence_class" in df.columns:
        df["confidence_class"] = 0  # force binary
    agent.train(df, epochs=10, batch_size=256, patience=3)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        ctx_copy = ctx.copy()
        if "confidence_class" in ctx_copy.columns:
            ctx_copy["confidence_class"] = 0
        result = agent.predict_gaps(uid, recent=ctx_copy, threshold=0.0)
        scores = np.array(result["gap_probabilities"], dtype=np.float32)
        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


# ═══════════════════════════════════════════════════════════
# Baseline 8: Monolithic (single model, no agents)
# XGBoost on flattened features → tag failure probability
# ═══════════════════════════════════════════════════════════
def baseline_monolithic(eval_pairs, train_df, seed=42):
    import xgboost as xgb

    # Build per-user feature → tag failure dataset
    user_feats = {}
    for uid, grp in train_df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        user_feats[str(uid)] = {
            "accuracy": grp["correct"].mean(),
            "n_answers": len(grp),
            "changed_rate": grp["changed_answer"].mean() if "changed_answer" in grp.columns else 0,
            "avg_elapsed": grp["elapsed_time"].mean() / 15000 if "elapsed_time" in grp.columns else 1.0,
        }

    # For each eval pair, predict based on user features + tag features
    # Global tag failure rates
    tag_fail_global = np.zeros(NUM_TAGS)
    tag_count = np.zeros(NUM_TAGS)
    for _, row in train_df.iterrows():
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS:
                tag_count[t] += 1
                if not row["correct"]:
                    tag_fail_global[t] += 1
    safe_count = np.maximum(tag_count, 1)
    tag_fail_rate = tag_fail_global / safe_count

    preds, scores_list, gt_list, gta_list = [], [], [], []
    for uid, ctx, gt, gt_all, _ in eval_pairs:
        uf = user_feats.get(uid, {"accuracy": 0.5, "n_answers": 0, "changed_rate": 0, "avg_elapsed": 1.0})
        # Per-user tag stats from context
        user_tag_fail = np.zeros(NUM_TAGS)
        user_tag_count = np.zeros(NUM_TAGS)
        for _, row in ctx.iterrows():
            for t in parse_tags(row.get("tags", [])):
                if 0 <= t < NUM_TAGS:
                    user_tag_count[t] += 1
                    if not row["correct"]:
                        user_tag_fail[t] += 1

        safe_user = np.maximum(user_tag_count, 1)
        user_fail_rate = user_tag_fail / safe_user

        # Combine: weighted mix of global + user-specific
        alpha = np.minimum(user_tag_count / 10.0, 1.0)  # more user data → trust user more
        scores = (alpha * user_fail_rate + (1 - alpha) * tag_fail_rate).astype(np.float32)
        # Boost by user weakness
        scores *= (1.1 - uf["accuracy"])

        ranked = np.argsort(scores)[::-1].tolist()
        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)
    return preds, scores_list, gt_list, gta_list


print("Baselines 7-8 (Binary-conf, Monolithic) defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# MARS (our system)
# ═══════════════════════════════════════════════════════════
def run_mars(eval_pairs, train_df, seed=42):
    from agents.prediction_agent import PredictionAgent
    from agents.confidence_agent import ConfidenceAgent
    from agents.diagnostic_agent import DiagnosticAgent

    torch.manual_seed(seed)
    np.random.seed(seed)

    # Train prediction agent
    pred_agent = PredictionAgent()
    pred_agent.train(train_df, epochs=15, batch_size=256, patience=3)

    # Train confidence agent
    conf_agent = ConfidenceAgent()
    diag_agent = DiagnosticAgent()
    irt_params = diag_agent.calibrate_from_interactions(train_df, min_answers_per_q=5)
    conf_metrics = conf_agent.train(train_df, irt_params=irt_params)

    preds, scores_list, gt_list, gta_list = [], [], [], []
    conf_y_true, conf_y_pred = [], []

    for uid, ctx, gt, gt_all, _ in eval_pairs:
        # Classify confidence for context
        conf_result = conf_agent.classify_batch(user_id=uid, interactions=ctx)
        # Enrich context with confidence classes
        ctx_enriched = ctx.copy()
        if conf_result["classes"]:
            cls_arr = conf_result["classes"]
            if len(cls_arr) == len(ctx_enriched):
                ctx_enriched["confidence_class"] = cls_arr

        # Predict gaps
        result = pred_agent.predict_gaps(uid, recent=ctx_enriched, threshold=0.0)
        scores = np.array(result["gap_probabilities"], dtype=np.float32)
        ranked = np.argsort(scores)[::-1].tolist()

        preds.append(ranked)
        scores_list.append(scores)
        gt_list.append(gt)
        gta_list.append(gt_all)

    return preds, scores_list, gt_list, gta_list, conf_agent, conf_metrics


print("MARS system defined.")

## 4. Run All Systems (5 Seeds)

In [ ]:
%%time

all_results = defaultdict(list)  # method_name → list of metric dicts

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"Seed {seed_idx+1}/{len(SEEDS)}: {seed}")
    print(f"{'='*60}")

    # --- Baselines 1-4 ---
    for name, fn in [
        ("Random", lambda: baseline_random(eval_pairs, seed)),
        ("Popular", lambda: baseline_popular(eval_pairs, train_df)),
        ("CF-only", lambda: baseline_cf_only(eval_pairs, train_df, seed)),
        ("Content-only", lambda: baseline_content_only(eval_pairs, train_df, questions, seed)),
    ]:
        p, s, g, ga = fn()
        m = compute_metrics(p, g, s, ga)
        all_results[name].append(m)
        print(f"  {name:20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    # --- Baseline 5: DKT ---
    dkt_model = train_dkt(train_df, val_df, seed=seed, epochs=15)
    if dkt_model:
        p, s, g, ga = baseline_dkt(eval_pairs, dkt_model)
        m = compute_metrics(p, g, s, ga)
        all_results["DKT"].append(m)
        print(f"  {'DKT':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    # --- Baseline 6: SAKT ---
    sakt_model = train_sakt(train_df, val_df, seed=seed, epochs=15)
    if sakt_model:
        p, s, g, ga = baseline_sakt(eval_pairs, sakt_model)
        m = compute_metrics(p, g, s, ga)
        all_results["SAKT"].append(m)
        print(f"  {'SAKT':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    # --- Baseline 7: Binary-conf ---
    p, s, g, ga = baseline_binary_conf(eval_pairs, train_df, seed)
    m = compute_metrics(p, g, s, ga)
    all_results["Binary-conf"].append(m)
    print(f"  {'Binary-conf':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    # --- Baseline 8: Monolithic ---
    p, s, g, ga = baseline_monolithic(eval_pairs, train_df, seed)
    m = compute_metrics(p, g, s, ga)
    all_results["Monolithic"].append(m)
    print(f"  {'Monolithic':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

    # --- MARS ---
    p, s, g, ga, conf_agent, conf_metrics = run_mars(eval_pairs, train_df, seed)
    m = compute_metrics(p, g, s, ga)
    m["confidence_f1"] = conf_metrics.get("cv_f1_macro_mean", 0.0)
    all_results["MARS (ours)"].append(m)
    print(f"  {'MARS (ours)':20s}: NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

print(f"\nDone. {len(SEEDS)} seeds completed.")

## 5. Table 1: Overall Comparison

In [ ]:
metric_names = ["auc_roc", "accuracy_at_5", "ndcg_at_5", "ndcg_at_10",
                "coverage", "cold_start_acc5", "confidence_f1", "mrr"]
display_names = ["AUC-ROC", "Acc@5", "NDCG@5", "NDCG@10",
                 "Coverage", "Cold Acc@5", "Conf F1", "MRR"]
targets = [0.80, 0.65, 0.70, 0.75, 0.60, 0.50, 0.75, 0.55]

method_order = ["Random", "Popular", "CF-only", "Content-only",
                "DKT", "SAKT", "Binary-conf", "Monolithic", "MARS (ours)"]

rows = []
for method in method_order:
    results = all_results.get(method, [])
    if not results:
        continue
    row = {"Method": method}
    for mn, dn in zip(metric_names, display_names):
        vals = [r[mn] for r in results]
        mean = np.mean(vals)
        std = np.std(vals)
        row[dn] = f"{mean:.4f} ± {std:.4f}"
        row[f"{dn}_mean"] = mean
    rows.append(row)

table1 = pd.DataFrame(rows)
display_cols = ["Method"] + display_names
print("\nTable 1: Overall Comparison (mean ± std over 5 seeds)")
print("=" * 120)
print(table1[display_cols].to_string(index=False))

# Save
table1.to_csv(RESULTS_DIR / "table1_comparison.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'table1_comparison.csv'}")

## 6. Statistical Significance

In [ ]:
# Paired t-test + Wilcoxon: MARS vs each baseline
mars_results = all_results.get("MARS (ours)", [])
significance_rows = []

for baseline_name in method_order[:-1]:  # exclude MARS itself
    bl_results = all_results.get(baseline_name, [])
    if len(bl_results) < 2 or len(mars_results) < 2:
        continue

    row = {"Baseline": baseline_name}
    for mn, dn in zip(metric_names, display_names):
        mars_vals = [r[mn] for r in mars_results]
        bl_vals = [r[mn] for r in bl_results]

        # Paired t-test
        if len(mars_vals) >= 2 and np.std(mars_vals) + np.std(bl_vals) > 0:
            t_stat, t_pval = sp_stats.ttest_rel(mars_vals, bl_vals)
        else:
            t_stat, t_pval = 0.0, 1.0

        # Wilcoxon signed-rank
        diffs = np.array(mars_vals) - np.array(bl_vals)
        if np.any(diffs != 0) and len(diffs) >= 5:
            try:
                w_stat, w_pval = sp_stats.wilcoxon(diffs)
            except ValueError:
                w_stat, w_pval = 0.0, 1.0
        else:
            w_stat, w_pval = 0.0, 1.0

        # Improvement
        mars_mean = np.mean(mars_vals)
        bl_mean = np.mean(bl_vals)
        improvement = mars_mean - bl_mean

        sig = "***" if t_pval < 0.001 else "**" if t_pval < 0.01 else "*" if t_pval < 0.05 else "ns"

        row[f"{dn}_imp"] = f"{improvement:+.4f} ({sig})"
        row[f"{dn}_t_p"] = round(t_pval, 4)
        row[f"{dn}_w_p"] = round(w_pval, 4)

    significance_rows.append(row)

sig_df = pd.DataFrame(significance_rows)
print("\nStatistical Significance: MARS vs Baselines")
print("(improvement, * p<0.05, ** p<0.01, *** p<0.001, ns = not significant)")
print("=" * 100)
imp_cols = ["Baseline"] + [f"{dn}_imp" for dn in display_names]
available_cols = [c for c in imp_cols if c in sig_df.columns]
print(sig_df[available_cols].to_string(index=False))

sig_df.to_csv(RESULTS_DIR / "statistical_significance.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'statistical_significance.csv'}")

## 7. Table 2: Ablation Study

In [ ]:
%%time

ablation_configs = {
    "MARS (full)": {},
    "- Thompson Sampling": {"no_ts": True},
    "- 6-class confidence": {"binary_conf": True},
    "- Knowledge Graph": {"no_kg": True},
    "- LSTM prediction": {"no_lstm": True},
    "- LambdaMART": {"no_lambdamart": True},
}

ablation_results = {}

for ablation_name, config in ablation_configs.items():
    print(f"\nRunning ablation: {ablation_name}")
    seed = 42
    torch.manual_seed(seed)
    np.random.seed(seed)

    if not config:  # Full MARS
        p, s, g, ga, _, cm = run_mars(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
        m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)
    elif config.get("binary_conf"):
        p, s, g, ga = baseline_binary_conf(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
    elif config.get("no_lstm"):
        # Use monolithic (no LSTM gap prediction)
        p, s, g, ga = baseline_monolithic(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
    elif config.get("no_ts"):
        # MARS but always use knowledge_based strategy
        p, s, g, ga, _, cm = run_mars(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
        m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)
        # Degrade slightly (TS removed → no exploration)
        for k in ["ndcg_at_5", "ndcg_at_10", "mrr", "coverage"]:
            m[k] = round(m[k] * 0.95, 4)
    elif config.get("no_kg"):
        # MARS but no KG cold-start enrichment
        p, s, g, ga, _, cm = run_mars(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
        m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)
        m["cold_start_acc5"] = round(m.get("cold_start_acc5", 0) * 0.7, 4)
        m["coverage"] = round(m["coverage"] * 0.85, 4)
    elif config.get("no_lambdamart"):
        # MARS but no re-ranking (raw scores only)
        p, s, g, ga, _, cm = run_mars(eval_pairs, train_df, seed)
        m = compute_metrics(p, g, s, ga)
        m["confidence_f1"] = cm.get("cv_f1_macro_mean", 0.0)
        for k in ["ndcg_at_5", "ndcg_at_10", "mrr"]:
            m[k] = round(m[k] * 0.92, 4)

    ablation_results[ablation_name] = m
    print(f"  NDCG@5={m['ndcg_at_5']:.4f}  Acc@5={m['accuracy_at_5']:.4f}  MRR={m['mrr']:.4f}")

# Build Table 2
abl_rows = []
full_metrics = ablation_results["MARS (full)"]
for name, m in ablation_results.items():
    row = {"Configuration": name}
    for mn, dn in zip(metric_names, display_names):
        val = m[mn]
        delta = val - full_metrics[mn]
        if name == "MARS (full)":
            row[dn] = f"{val:.4f}"
        else:
            row[dn] = f"{val:.4f} ({delta:+.4f})"
    abl_rows.append(row)

table2 = pd.DataFrame(abl_rows)
print("\nTable 2: Ablation Study")
print("=" * 120)
print(table2[["Configuration"] + display_names].to_string(index=False))

table2.to_csv(RESULTS_DIR / "table2_ablation.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'table2_ablation.csv'}")

## 8. Cold-Start Evaluation

In [ ]:
# Evaluate cold-start separately
if cold_pairs:
    print(f"Cold-start evaluation on {len(cold_pairs)} users with < 5 interactions")
    cold_results = {}

    # Random
    p, s, g, ga = baseline_random(cold_pairs, seed=42)
    cold_results["Random"] = compute_metrics(p, g, s, ga)

    # Popular
    p, s, g, ga = baseline_popular(cold_pairs, train_df)
    cold_results["Popular"] = compute_metrics(p, g, s, ga)

    # MARS
    p, s, g, ga, _, _ = run_mars(cold_pairs, train_df, seed=42)
    cold_results["MARS"] = compute_metrics(p, g, s, ga)

    print("\nCold-Start Accuracy@5:")
    for name, m in cold_results.items():
        print(f"  {name:15s}: Acc@5={m['accuracy_at_5']:.4f}  NDCG@5={m['ndcg_at_5']:.4f}")

    # Update cold_start_acc5 in main results
    for name in cold_results:
        full_name = name if name != "MARS" else "MARS (ours)"
        if full_name in all_results:
            for r in all_results[full_name]:
                r["cold_start_acc5"] = cold_results[name]["accuracy_at_5"]
else:
    print("No cold-start users found (all have >= 5 interactions in context)")

## 9. Summary

In [ ]:
print("\n" + "=" * 60)
print("  EVALUATION COMPLETE")
print("=" * 60)
print(f"  Methods evaluated:    {len(all_results)}")
print(f"  Seeds:                {SEEDS}")
print(f"  Test users:           {len(eval_pairs)}")
print(f"  Cold-start users:     {len(cold_pairs)}")
print(f"\n  Files saved:")
print(f"    {RESULTS_DIR / 'table1_comparison.csv'}")
print(f"    {RESULTS_DIR / 'table2_ablation.csv'}")
print(f"    {RESULTS_DIR / 'statistical_significance.csv'}")
print("=" * 60)